# CloudServe ticket walkthrough
Runs the remediated LangGraph workflow offline (no model, no key) on the demo tickets and shows the audit record. Run from the repository root.

In [ ]:
import json, os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
from cloudserve.classifier import NaiveBayesIntentClassifier
from cloudserve.graph import DecisionEngine
from cloudserve.pipeline import KnowledgeBase
from cloudserve.routing import load_policy
engine = DecisionEngine(KnowledgeBase.from_path('data/documentation.json'), load_policy(),
                        NaiveBayesIntentClassifier.from_development_file('data/development_tickets.json'),
                        auto_send=True)
engine.implementation

In [ ]:
tickets = json.load(open('examples/four_channels.json')) + [json.load(open('examples/guardrail_ticket.json'))]
for r in map(engine.process, tickets):
    print(f"{r['ticket_id']:<14} {r['channel']:<6} {r['prediction']['value']:<24} {r['action_taken']:<13} {r['reason_codes']}")

In [ ]:
rec = engine.process(tickets[0])
print(rec['response'])
{k: rec[k] for k in ('action_taken', 'status', 'citations', 'threshold_applied', 'guardrail_results', 'prompt_version', 'policy_version')}

Evaluation over a full file: `python -m cloudserve.evaluation --input data/validation_tickets.json --output evaluation/results/validation_offline_default`.